In [ ]:
# Multi-Commodity Network Flow (4 nodes, 4 arcs, 2 commodities) — PuLP
import pulp as pl
nodes = [1, 2, 3, 4]
arcs  = [(1, 2), (1, 3), (2, 4), (3, 4)]
K     = [1, 2]                      # commodities
cap   = {(1,2):20, (1,3):15, (2,4):25, (3,4):20}
cost  = {(1,2,1):2,(1,2,2):3,(1,3,1):3,(1,3,2):2,
         (2,4,1):1,(2,4,2):2,(3,4,1):2,(3,4,2):1}
dem   = {(1,1):-10,(1,2):-5,(2,1):0,(2,2):0,
         (3,1):0,(3,2):0,(4,1):10,(4,2):5}   # <0 supply, >0 demand

prob = pl.LpProblem("MultiCommodityFlow", pl.LpMinimize)
f = pl.LpVariable.dicts("f", [(i,j,k) for (i,j) in arcs for k in K], lowBound=0)
prob += pl.lpSum(cost[i,j,k]*f[(i,j,k)] for (i,j) in arcs for k in K), "Total_Cost"
# shared arc capacity
for (i,j) in arcs:
    prob += pl.lpSum(f[(i,j,k)] for k in K) <= cap[(i,j)], f"Cap_{i}_{j}"
# flow conservation per node, per commodity
for n in nodes:
    for k in K:
        prob += (pl.lpSum(f[(i,j,k)] for (i,j) in arcs if i==n)
                 - pl.lpSum(f[(i,j,k)] for (i,j) in arcs if j==n)
                 == -dem[(n,k)]), f"Bal_{n}_{k}"
prob.solve()
print("Status:", pl.LpStatus[prob.status])
for (i,j) in arcs:
    print(f"arc {i}->{j}: " + ", ".join(f"k{k}={f[(i,j,k)].value():g}" for k in K))
print("Total cost =", pl.value(prob.objective))
